# WRT-Omega — Parameter Golf Training

**Runtime required**: GPU → A100 (80GB)  
**Target**: 10-min wallclock run, val_bpb < 1.2244 (baseline)  
**Model**: 16.5M params, 768-dim, 8 effective layers (2 stem + 5 recurrent loops + 1 crown), 4 workspace tokens

In [ ]:
# Cell 1 — Verify GPU
!nvidia-smi | head -20

In [ ]:
# Cell 2 — Install deps
!pip install -q sentencepiece huggingface_hub

In [ ]:
# Cell 3 — Clone repo
import os
if not os.path.exists('parameter-golf'):
    !git clone https://github.com/JairoPolanco/parameter-golf.git
os.chdir('parameter-golf')
!git log --oneline -3

In [ ]:
# Cell 4 — Download data
# Downloads the val split + 10 train shards (~1B tokens, enough for the 10-min run).
# Each shard = 100M tokens. 10 min @ ~18k steps * 524k tokens/step = ~9.4B tokens,
# but the loader wraps around shards, so 10 shards is plenty.
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 10 2>&1

In [ ]:
# Cell 5 — Verify data
import glob
train_files = sorted(glob.glob('data/datasets/fineweb10B_sp1024/fineweb_train_*.bin'))
val_files   = sorted(glob.glob('data/datasets/fineweb10B_sp1024/fineweb_val_*.bin'))
tok_file    = 'data/tokenizers/fineweb_1024_bpe.model'
print(f'Train shards : {len(train_files)}')
print(f'Val shards   : {len(val_files)}')
print(f'Tokenizer OK : {os.path.exists(tok_file)}')
assert len(train_files) > 0 and len(val_files) > 0 and os.path.exists(tok_file), 'Data missing!'

In [ ]:
# Cell 6 — Sanity check: model init + forward pass
import sys, torch
sys.path.insert(0, '.')
from train_gpt import GPT, Hyperparameters

args = Hyperparameters()
device = torch.device('cuda')
model = GPT(
    vocab_size=args.vocab_size, stem_layers=args.stem_layers,
    n_loops=args.n_loops, crown_layers=args.crown_layers,
    workspace_tokens=args.workspace_tokens, model_dim=args.model_dim,
    num_heads=args.num_heads, num_kv_heads=args.num_kv_heads,
    mlp_mult=args.mlp_mult, tie_embeddings=args.tie_embeddings,
    tied_embed_init_std=args.tied_embed_init_std, logit_softcap=args.logit_softcap,
    rope_base=args.rope_base, qk_gain_init=args.qk_gain_init,
    workspace_diversity_weight=args.workspace_diversity_weight,
    seq_len=args.train_seq_len,
).to(device).bfloat16()

n_params = sum(p.numel() for p in model.parameters())
print(f'Params : {n_params:,}  (~{n_params*0.93/1e6:.1f} MB compressed)')
assert 15_000_000 < n_params < 18_000_000, f'Unexpected param count: {n_params}'

x = torch.randint(0, 1024, (2, 1024), device=device)
with torch.autocast('cuda', torch.bfloat16):
    loss = model(x, x)
print(f'Forward pass OK — loss: {loss.item():.4f}')

In [ ]:
# Cell 7 — TRAIN (10-minute wallclock cap, single A100)
#
# Key env vars (all have sane defaults, override here if needed):
#   MAX_WALLCLOCK_SECONDS=600   <- 10-minute hard cap
#   ITERATIONS=20000            <- max steps (usually wallclock cap fires first)
#   WARMUP_STEPS=20             <- JIT compile warmup steps
#   VAL_LOSS_EVERY=1000         <- validate every N steps
#   TRAIN_LOG_EVERY=200         <- log train loss every N steps
#
# WORLD_SIZE=1 → grad_accum_steps = 8 (same total tokens/step as 8-GPU baseline)

!MAX_WALLCLOCK_SECONDS=600 torchrun --standalone --nproc_per_node=1 train_gpt.py 2>&1

In [ ]:
# Cell 8 — Show results
import glob, os
logs = sorted(glob.glob('logs/*.txt'), key=os.path.getmtime)
assert logs, 'No log file found'
with open(logs[-1]) as f:
    lines = f.readlines()

# Skip the source-code header; print from the === separator onward
sep_count = 0
for line in lines:
    if '=' * 40 in line:
        sep_count += 1
    if sep_count >= 2:  # second separator = end of source dump
        print(line, end='')

In [ ]:
# Cell 9 — Artifact size check
import os
model_bytes = os.path.getsize('final_model.int8.ptz')
code_bytes  = len(open('train_gpt.py', 'rb').read())
total_mb    = (model_bytes + code_bytes) / 1e6
print(f'Model (int8+zlib) : {model_bytes/1e6:.3f} MB')
print(f'Code              : {code_bytes/1e3:.1f} KB')
print(f'Total             : {total_mb:.3f} MB  (limit: 16 MB)')
print(f'Under limit       : {total_mb < 16.0}')